# Load Modules

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report

In [ ]:
import os
train_path = '/kaggle/input/brain-tumor-classification-mri/Training'
classes = {'no_tumor': 0, 'pituitary_tumor': 1}

#Load and preprocess Training data

In [ ]:
import cv2
X = []
Y = []

for cls in classes:
    class_path = os.path.join(train_path, cls)
    for img_name in os.listdir(class_path):
        img = cv2.imread(os.path.join(class_path, img_name), 0)
        img = cv2.resize(img, (200, 200))
        X.append(img)
        Y.append(classes[cls])

In [ ]:
np.unique(Y)

In [ ]:
X = np.array(X)
Y = np.array(Y)
X_updated = X.reshape(len(X), -1)

#Flatten images

In [ ]:
pd.Series(Y).value_counts()

In [ ]:
X.shape, X_updated.shape

#Visualize the data

In [ ]:
plt.imshow(X[0], cmap='gray')

In [ ]:
X_updated = X.reshape(len(X), -1)
X_updated.shape

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(X_updated, Y, random_state=10,
                                               test_size=20)

In [ ]:
xtrain.shape, xtest.shape

In [ ]:
print(xtrain.max(), xtrain.min())
print(xtest.max(), xtest.min())
xtrain = xtrain/255
xtest = xtest/255
print(xtrain.max(), xtrain.min())
print(xtest.max(), xtest.min())

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca_train = xtrain
pca_test = xtest

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [ ]:
import warnings
warnings.filterwarnings('ignore')

lg = LogisticRegression(C=0.1)
lg.fit(xtrain, ytrain)

In [ ]:
sv = SVC()
sv.fit(xtrain, ytrain)

In [ ]:
print("Training Score:", lg.score(xtrain, ytrain))
print("Testing Score:", lg.score(xtest, ytest))

In [ ]:

print("Training Score:", sv.score(xtrain, ytrain))
print("Testing Score:", sv.score(xtest, ytest))

# Testing

In [ ]:
test_path = '/kaggle/input/brain-tumor-classification-mri/Testing'
dec = {0:'No Tumor', 1:'Positive Tumor'}

In [ ]:
plt.figure(figsize=(12, 8))
c = 1
for i in os.listdir(os.path.join(test_path, 'no_tumor'))[:9]:
    plt.subplot(3, 3, c)
    img = cv2.imread(os.path.join(test_path, 'no_tumor', i), 0)
    img_resized = cv2.resize(img, (200, 200))
    img_flattened = img_resized.reshape(1, -1) / 255
    prediction = sv.predict(img_flattened)
    plt.title(dec[prediction[0]])
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    c += 1
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
c = 1
for i in os.listdir(os.path.join(test_path, 'no_tumor'))[:16]:
    plt.subplot(4, 4, c)
    img = cv2.imread(os.path.join(test_path, 'no_tumor', i), 0)
    img_resized = cv2.resize(img, (200, 200))
    img_flattened = img_resized.reshape(1, -1) / 255
    prediction = sv.predict(img_flattened)
    plt.title(dec[prediction[0]])
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    c += 1
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV
model = LogisticRegression(max_iter=1000)

param_grid = {
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1, 10, 100],         # Inverse of regularization strength
    'solver': ['liblinear', 'saga'],      # Compatible solvers
    'l1_ratio': [0, 0.5, 1]               # Only used if penalty='elasticnet'
}


In [ ]:
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1, error_score='raise')
grid_search.fit(xtrain, ytrain)

In [ ]:
print("Best Parameters:", grid_search.best_params_)
print("Best Score (CV):", grid_search.best_score_)

# Evaluate on test set
y_pred = grid_search.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
from pycaret.classification import *

In [ ]:
exp1 = setup(x, target= y, session_id=123)